# v6 — QLoRA: LoRA on a 4-bit frozen base
The base model is loaded in 4-bit (NF4) and frozen; only the LoRA matrices train in fp16.
This is what makes billion-parameter models tunable on small GPUs. On a 125M model expect similar accuracy,
lower memory and somewhat slower steps — which is itself the finding. Runtime → T4 GPU → Run all. ~1.5 h.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v6_qlora_4bit'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v6_qlora_4bit'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v6_qlora_4bit/results'
!pip -q install -U transformers datasets accelerate scikit-learn peft bitsandbytes
# Colab ships torchao 0.10, which peft rejects while scanning layer types; we do not use it
!pip uninstall -y -q torchao 2>/dev/null; echo removed torchao

In [ ]:
!python prepare_data.py --classes 5 --per-class 40000

In [ ]:
# ~6 min: peak GPU memory for QLoRA, to compare with v5's memory_full_finetune / memory_lora
!python finetune_roberta.py --name memory_qlora --data-dir data5 --batch-size 32 --limit 3200 --epochs 1 --eval-steps 100 --qlora --results-dir {RESULTS} --ckpt-dir /content/memory_qlora
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
print(f"QLoRA: peak GPU {m['memory_qlora'].get('peak_gpu_gb')} GB, trainable params {m['memory_qlora'].get('trainable_params'):,}")

In [ ]:
# ~1-1.5 h
!python finetune_roberta.py --name roberta_base_5class_qlora --data-dir data5 --batch-size 32 --eval-steps 2500 --qlora --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_base_5class_qlora/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['roberta_base_5class_qlora', 'roberta_base_5class_qlora_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!zip -qr btp_v6_qlora_4bit.zip models/roberta_base_5class_qlora versions/v6_qlora_4bit/results && ls -lh btp_v6_qlora_4bit.zip
!cp btp_v6_qlora_4bit.zip {DRIVE}/
from google.colab import files
files.download('btp_v6_qlora_4bit.zip')